# 07 — Three-Engine Evidence Tree & Explain Demo

One schema, three engines, shared Store. All output comes from real framework calls.

**What this demonstrates:**
1. **Evidence tree structure** — full node hierarchy walk (candidate_result → support → witness → assertion)
2. **Certainty propagation** — condition_weights + confidence → bottleneck vs additive comparison
3. **ProbLog proof tree** — proof_goal / proof_leaf nodes + probability pipeline
4. **PyReason timeline** — propagation chains + events + bounds
5. **Cross-engine data flow** — accepted facts from one engine feed the next
6. **Full explain pipeline** for each: tree/timeline → summary → narrative → NL

ProbLog and PyReason use mocked runners; the entire evaluate → accept → explain pipeline is real.

**Prerequisites:** [01](01_sdk_basics.ipynb)–[02](02_rules_and_derivations.ipynb) for SDK basics.

## 0. Imports

In [ ]:
import sys, tempfile
from pathlib import Path
from pprint import pprint
from unittest.mock import patch

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

from factpy_kernel.sdk import SDKStore, Entity, Identity, Field, Rule, Pred, vars as sdk_vars
from factpy_kernel.sdk.compile import compile_schema_from_classes
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session, close_runtime_session, reset_runtime_sessions_for_tests,
    write_runtime_fact, evaluate_runtime_derivation, accept_runtime_derivation,
    explain_runtime_tree, explain_runtime_summary, explain_runtime_narrative, explain_runtime_nl,
    explain_runtime_timeline, explain_runtime_timeline_summary, explain_runtime_timeline_narrative,
)
import factpy_kernel.adapters.problog
import factpy_kernel.adapters.pyreason
from factpy_kernel.adapters.pyreason.session import PyReasonSession
from factpy_kernel.adapters.pyreason.runner import PyReasonRunConfig, PyReasonRunResult

In [ ]:
def print_tree(node, indent=0):
    prefix = "  " * indent
    kind = node.get("node_kind", "?")
    label = kind
    if kind == "candidate_result":
        rk = node.get("root_result_kind", "?")
        label += f"  (root_result_kind={rk})"
        em = node.get("engine_meta")
        if em:
            label += f"  engine_meta={em}"
    elif kind == "predicate_witness_group":
        label += f"  pred_id={node.get('pred_id', '?')}"
        cc = node.get("condition_confidence")
        if cc is not None:
            label += f"  condition_confidence={cc}"
    elif kind == "assertion_fact":
        claims = node.get("claim_args", [])
        label += f"  [{', '.join(c.get('val','?') for c in claims)}]"
        conf = node.get("confidence")
        if conf is not None:
            label += f"  confidence={conf}"
    elif kind == "proof_goal":
        label += f"  pred_id={node.get('pred_id', '?')}  goal_args={node.get('goal_args', [])}"
    elif kind == "proof_leaf":
        label += f"  pred_id={node.get('pred_id', '?')}  goal_args={node.get('goal_args', [])}"
    elif kind == "rule_ref":
        label += f"  {node.get('rule_ref_id', '?')} v{node.get('rule_ref_version', '?')}"
    elif kind == "non_fact_check":
        label += f"  {node.get('check_kind', '?')}  status={node.get('status', '?')}"
    print(f"{prefix}- {label}")
    for child in node.get("children", []):
        print_tree(child, indent + 1)

## 1. Schema + Session Setup

All three engines operate on the same `Researcher` entity.

In [ ]:
class Researcher(Entity):
    researcher_id: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")
    expertise: str = Field(cardinality="single")
    impact_score: str = Field(cardinality="single")
    tag_seed: str = Field(cardinality="single")
    tag: str = Field(cardinality="single")
    risk_flag: str = Field(cardinality="single")

schema_ir = compile_schema_from_classes([Researcher])
sdk = SDKStore([Researcher], schema_ir=schema_ir)

# Rule with condition_weights → enables certainty routing
with sdk_vars("r", "exp", "score") as (r, exp, score):
    qualified_rule = Rule(
        id="q.qualified", version="1.0.0",
        select=[r, exp],
        where=[Pred("researcher:expertise", r, exp), Pred("researcher:impact_score", r, score)],
        expose=True,
        condition_weights={"b0.a0": 0.8, "b0.a1": 0.5},
    )

# Rule WITHOUT condition_weights → no certainty
with sdk_vars("r", "exp", "score") as (r, exp, score):
    plain_rule = Rule(
        id="q.plain", version="1.0.0",
        select=[r, exp],
        where=[Pred("researcher:expertise", r, exp), Pred("researcher:impact_score", r, score)],
        expose=True,
    )

# Registry
registry_dir = tempfile.mkdtemp(prefix="three_engine_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
registry.register_rule_spec(sdk._compile_rule_input(qualified_rule))
registry.register_rule_spec(sdk._compile_rule_input(plain_rule))

# Session
reset_runtime_sessions_for_tests()
resp = open_runtime_session({"registry_root": registry_dir})
session_id = resp["session"]["session_id"]

# Seed facts with confidence values
alice_ref = sdk.ref(Researcher, researcher_id="Alice")
bob_ref = sdk.ref(Researcher, researcher_id="Bob")

for ref, name, expertise, score, exp_conf, score_conf in [
    (alice_ref, "Alice Chen", "NLP", "92", 0.95, 0.7),
    (bob_ref, "Bob Zhang", "CV", "85", 0.88, 0.6),
]:
    write_runtime_fact(session_id, {"pred_id": "researcher:name", "e_ref": ref,
        "rest_terms": [["string", name]]}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:expertise", "e_ref": ref,
        "rest_terms": [["string", expertise]], "meta": {"confidence": exp_conf}}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:impact_score", "e_ref": ref,
        "rest_terms": [["string", score]], "meta": {"confidence": score_conf}}, kind="add")
    write_runtime_fact(session_id, {"pred_id": "researcher:tag_seed", "e_ref": ref,
        "rest_terms": [["string", "senior"]], "meta": {"confidence": 1.0}}, kind="add")

print(f"Session ready. Facts seeded:")
print(f"  Alice: expertise=NLP(conf=0.95), impact=92(conf=0.7), tag_seed=senior")
print(f"  Bob:   expertise=CV(conf=0.88),  impact=85(conf=0.6), tag_seed=senior")

---
## 2. Native Engine — Evidence Tree with Certainty

`condition_weights` on the rule + `confidence` on facts → certainty auto-routing.

### 2.1 Evaluate + Accept

In [ ]:
eval_native = evaluate_runtime_derivation(session_id, {"derivation": {
    "derivation_id": "drv.qualified", "version": "1.0.0",
    "target": "researcher:expertise", "head_vars": ["$r", "$exp"],
    "where": [["ruleref", "q.qualified", "1.0.0", ["$r", "$exp"]]],
    "mode": "native",
}})

native_cands = eval_native["evaluation"]["candidates"]
for c in native_cands:
    accept_runtime_derivation(session_id, {"candidate": c})

cid_native = native_cands[0]["candidate_id"]
print(f"Candidates: {len(native_cands)}")
print(f"  confidence_kind: {native_cands[0]['confidence_kind']} ← auto-routed because rule has condition_weights")

### 2.2 Evidence Tree — Node Hierarchy

The full tree structure: `candidate_result` → `support_section` → `predicate_witness_group` → `assertion_fact`.
Each witness group carries `condition_confidence` from the fact's `meta.confidence`.

In [ ]:
tree_resp = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_native})

print("=== Evidence Tree (Native) ===")
print_tree(tree_resp["tree"]["root"])

### 2.3 Certainty Summary — Bottleneck vs Additive

Two aggregation strategies:
- **Bottleneck**: `aggregate = min(condition_impacts)` — weakest link dominates
- **Additive**: `aggregate = sum(weighted_contributions)` — highlights relative importance

In [ ]:
summary_bn = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_native})
summary_add = explain_runtime_summary(session_id, {
    "kind": "candidate", "id": cid_native, "certainty_aggregation": "additive",
})

cs_bn = summary_bn.get("certainty_summary")
cs_add = summary_add.get("certainty_summary")

if cs_bn and cs_add:
    print(f"{'':25s} {'Bottleneck':>12s}  {'Additive':>12s}")
    print(f"{'Aggregate':25s} {cs_bn['aggregate_certainty']:>12}  {cs_add['aggregate_certainty']:>12}")
    for bn_c, ad_c in zip(cs_bn["conditions"], cs_add["conditions"]):
        print(f"  {bn_c['atom_key']:23s} {bn_c['impact']:>12}  {ad_c['impact']:>12}")
    print(f"{'Strategy':25s} {'min(impacts)':>12s}  {'sum(contribs)':>12s}")

### 2.4 Narrative + NL

In [ ]:
narrative_native = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_native})
nl_native = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_native})

n = narrative_native["narrative"]
print(f"Headline: {n['headline']}")
print(f"\nCertainty lines:")
for line in n.get("certainty_lines", []):
    print(f"  {line}")
bottleneck = n.get("certainty_bottleneck")
if bottleneck:
    print(f"  bottleneck: {bottleneck}")

print(f"\nNL ({len(nl_native['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl_native["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p}")

### 2.5 Negative Case — No `condition_weights`

Without `condition_weights`, certainty routing is **not activated**.

In [ ]:
eval_plain = evaluate_runtime_derivation(session_id, {"derivation": {
    "derivation_id": "drv.plain", "version": "1.0.0",
    "target": "researcher:expertise", "head_vars": ["$r", "$exp"],
    "where": [["ruleref", "q.plain", "1.0.0", ["$r", "$exp"]]],
    "mode": "native",
}})

plain_cand = eval_plain["evaluation"]["candidates"][0]
accept_runtime_derivation(session_id, {"candidate": plain_cand})

plain_summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": plain_cand["candidate_id"]})
print(f"confidence_kind: {plain_cand['confidence_kind']}")
print(f"certainty_summary: {plain_summary.get('certainty_summary')}")
print(">>> No condition_weights → no certainty routing → no certainty delivery")

---
## 3. ProbLog Engine — Proof Tree with Probability

ProbLog reads **facts already in the session** (same `tag_seed` facts).
Runner mocked; evaluate → accept → explain pipeline is real.

Returns CandidateEvidenceTree with `proof_goal`/`proof_leaf` nodes (not witness nodes).

In [ ]:
def _mock_problog_output():
    alice = sdk.ref(Researcher, researcher_id="Alice")
    return "\n".join([
        " call query(X1,X2) {0.00000} []",
        f'  result query(X1,X2) ("senior","{alice}") {{{{}}}} {{0.00012}} []',
        " complete query(X1,X2) {0.00013} {0.00013} []",
        f' call answer("senior","{alice}") {{0.00019}} [at 4:7]',
        f'  call researcher__tag_seed("senior","{alice}") {{0.00026}} [at 3:9]',
        f'   result researcher__tag_seed("senior","{alice}") ("senior","{alice}") {{{{}}}} {{0.00038}} [at 3:9]',
        f'  complete researcher__tag_seed("senior","{alice}") {{0.00039}} {{0.00013}} []',
        f'  result answer("senior","{alice}") ("senior","{alice}") {{{{}}}} {{0.00060}} []',
        f' complete answer("senior","{alice}") {{0.00061}} {{0.00042}} []',
        "",
        f'answer("senior","{alice}"):\t0.85',
    ])

with patch("factpy_kernel.adapters.problog.engine_eval.run_problog") as mock_run:
    mock_run.return_value = _mock_problog_output()
    eval_prob = evaluate_runtime_derivation(session_id, {"derivation": {
        "derivation_id": "drv.problog_tag", "version": "1.0.0",
        "target": "researcher:tag", "head_vars": ["$u", "$tag"],
        "where": [["pred", "researcher:tag_seed", ["$u", "$tag"]]],
        "mode": "problog",
    }})

prob_cand = eval_prob["evaluation"]["candidates"][0]
accept_runtime_derivation(session_id, {"candidate": prob_cand})
cid_prob = prob_cand["candidate_id"]
print(f"ProbLog: candidate accepted (support_kind={prob_cand['support_kind']})")

### 3.1 Proof Tree — Node Hierarchy

Unlike native's `predicate_witness_group`/`assertion_fact`, ProbLog uses
`proof_goal` (intermediate proof step) and `proof_leaf` (terminal fact).
`proof_leaf` does NOT carry `asrt_id` — no dead links.

In [ ]:
tree_prob = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_prob})

print("=== Evidence Tree (ProbLog) ===")
print_tree(tree_prob["tree"]["root"])

print(f"\nroot.engine_meta.probability = {tree_prob['tree']['root']['engine_meta']['probability']}")

### 3.2 Summary + Narrative + NL (Probability Pipeline)

In [ ]:
summary_prob = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_prob})
narrative_prob = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_prob})
nl_prob = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_prob})

sp = summary_prob["summary"]
print(f"problog_probability = {sp.get('problog_probability')}")
print(f"proof_goal_count = {sp.get('proof_goal_count')}")
print(f"proof_leaf_count = {sp.get('proof_leaf_count')}")

np_ = narrative_prob["narrative"]
print(f"\nprobability_lines = {np_.get('probability_lines')}")

print(f"\nNL ({len(nl_prob['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl_prob["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p}")

---
## 4. PyReason Engine — Propagation Timeline

PyReason reads the **same session facts**. Runner mocked; pipeline is real.
Returns `CandidateProvenanceTimeline` — chains of bound-change events, NOT a tree.

In [ ]:
derived_session = PyReasonSession(schema_ir)
derived_session._write_node_fact_internal("researcher:risk_flag", alice_ref, "true", bound=[1.0, 1.0])

trace_dict = {
    "engine": "pyreason", "trace_type": "event_log", "timesteps": 2,
    "node_events": [{
        "time": 0, "fixpoint_op": 1,
        "component": alice_ref, "component_type": "node", "label": "risk_flag",
        "old_bound": [0.0, 1.0], "new_bound": [1.0, 1.0],
        "occurred_due_to": "seed_fact", "clause_groundings": [],
    }],
    "edge_events": [],
}

with patch("factpy_kernel.adapters.pyreason.engine_eval.run_pyreason") as mock_pr:
    mock_pr.return_value = PyReasonRunResult(
        interpretation=None, trace=None, trace_dict=trace_dict,
        derived_session=derived_session,
        config=PyReasonRunConfig(timesteps=2, atom_trace=True),
        elapsed_seconds=0.01,
    )
    eval_pr = evaluate_runtime_derivation(session_id, {"derivation": {
        "derivation_id": "drv.pyreason_risk", "version": "1.0.0",
        "target": "researcher:risk_flag", "head_vars": ["$r"],
        "where": [["pred", "researcher:expertise", ["$r", "$exp"]]],
        "mode": "pyreason",
    }})

pr_cand = eval_pr["evaluation"]["candidates"][0]
accept_runtime_derivation(session_id, {"candidate": pr_cand})
cid_pr = pr_cand["candidate_id"]

# explain-tree returns error (correct — PyReason is timeline, not tree)
tree_err = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_pr})
print(f"explain-tree: ok={tree_err['ok']} ← correct, PyReason is not tree")

### 4.1 Timeline Structure + Full Pipeline

In [ ]:
tl = explain_runtime_timeline(session_id, {"kind": "candidate", "id": cid_pr})
summary_pr = explain_runtime_timeline_summary(session_id, {"kind": "candidate", "id": cid_pr})
narrative_pr = explain_runtime_timeline_narrative(session_id, {"kind": "candidate", "id": cid_pr})
nl_pr = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_pr})

timeline = tl["timeline"]
print(f"=== CandidateProvenanceTimeline ===")
print(f"kind = {timeline['kind']}")
print(f"timesteps = {timeline['timesteps']}")
print(f"chains ({len(timeline['chains'])}):")
for chain in timeline["chains"]:
    print(f"  {chain['component']}.{chain['label']}:")
    for evt in chain["events"]:
        print(f"    t={evt['time']}: {evt['old_bound']} → {evt['new_bound']} by {evt['trigger']}")

s = summary_pr["summary"]
print(f"\nsummary: chains={s['chain_count']}, seeds={s['seed_count']}, timesteps={s['timesteps']}")

print(f"\nnarrative headline: {narrative_pr['narrative']['headline']}")

print(f"\nNL ({len(nl_pr['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl_pr["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p}")

---
## 5. Cross-Engine Comparison

In [ ]:
print("=" * 70)
print("THREE-ENGINE EXPLAIN COMPARISON")
print("=" * 70)

s_native = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_native})
cs = s_native.get("certainty_summary")
print(f"\n[Native] CandidateEvidenceTree")
print(f"  witnesses={s_native['summary']['witness_assertion_count']}, rule_refs={s_native['summary']['rule_ref_count']}")
if cs: print(f"  certainty={cs['aggregate_certainty']} ({cs['aggregation']})")

s_prob = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_prob})
print(f"\n[ProbLog] CandidateEvidenceTree (proof nodes)")
print(f"  proof_goals={s_prob['summary'].get('proof_goal_count',0)}, proof_leaves={s_prob['summary'].get('proof_leaf_count',0)}")
print(f"  probability={s_prob['summary'].get('problog_probability')}")

s_pr = explain_runtime_timeline_summary(session_id, {"kind": "candidate", "id": cid_pr})
print(f"\n[PyReason] CandidateProvenanceTimeline")
print(f"  chains={s_pr['summary']['chain_count']}, timesteps={s_pr['summary']['timesteps']}")
print(f"  final_bound={s_pr['summary'].get('final_bound')}")

print(f"\n{'=' * 70}")
print("Same schema, same session, same Store. Three explain surfaces.")
print(f"{'=' * 70}")

In [ ]:
close_runtime_session(session_id)
reset_runtime_sessions_for_tests()
print("Session closed.")

## Architecture Summary

```
  Shared Schema + Store (single session, single ledger)
       │
       ├── Native evaluate → accept → CandidateEvidenceTree
       │     predicate_witness_group / assertion_fact + certainty_summary
       │     condition_weights + confidence → bottleneck / additive
       │
       ├── ProbLog evaluate → accept → CandidateEvidenceTree
       │     proof_goal / proof_leaf + problog_probability
       │     probability flows: summary → narrative → NL
       │
       └── PyReason evaluate → accept → CandidateProvenanceTimeline
             chains + events + bounds (NOT tree)
             explain-timeline endpoints + polymorphic NL dispatch

  Cross-engine: accepted facts from Engine A visible to Engine B
  Unified interface, not unified implementation
```